In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType
from delta.tables import DeltaTable

CATALOG = "workspace"
SCHEMA = "oss_ecosystem"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_github_events"
SILVER_ISSUE_TABLE = f"{CATALOG}.{SCHEMA}.silver_issue_events"

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.oss_ecosystem.silver_issue_events (
    event_id STRING,
    event_ts TIMESTAMP,
    actor_id BIGINT,
    repo_id BIGINT,
    action STRING,
    issue_id BIGINT,
    issue_number BIGINT,
    state STRING,
    source_file STRING,
    source_hour TIMESTAMP,
    ingested_at TIMESTAMP
)
USING DELTA
""")

In [0]:
issue_payload_schema = StructType([
    StructField("action", StringType(), True),
    StructField(
        "issue",
        StructType([
            StructField("id", LongType(), True),
            StructField("number", LongType(), True),
            StructField("state", StringType(), True)
        ]),
        True
    )
])

In [0]:
issue_bronze_df = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("event_type") == "IssuesEvent")
)

print(
    "IssuesEvent rows in Bronze:",
    issue_bronze_df.count()
)

In [0]:
existing_issue_ids = (
    spark.table(SILVER_ISSUE_TABLE)
    .select("event_id")
)

new_issue_bronze_df = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("event_type") == "IssuesEvent")
    .join(
        existing_issue_ids,
        on="event_id",
        how="left_anti"
    )
)

print(
    "New IssuesEvents to transform:",
    new_issue_bronze_df.count()
)

In [0]:
silver_issue_df = (
    new_issue_bronze_df
    .withColumn(
        "payload_parsed",
        F.from_json(
            F.col("payload_json"),
            issue_payload_schema
        )
    )
    .select(
        "event_id",

        F.to_timestamp("created_at")
            .alias("event_ts"),

        F.get_json_object("actor_json", "$.id")
            .cast("long")
            .alias("actor_id"),

        F.get_json_object("repo_json", "$.id")
            .cast("long")
            .alias("repo_id"),

        F.col("payload_parsed.action")
            .alias("action"),

        F.col("payload_parsed.issue.id")
            .alias("issue_id"),

        F.col("payload_parsed.issue.number")
            .alias("issue_number"),

        F.col("payload_parsed.issue.state")
            .alias("state"),

        "source_file",
        "source_hour",
        "ingested_at"
    )
)

In [0]:
print(
    "Silver Issue rows:",
    silver_issue_df.count()
)

silver_issue_df.select(
    F.sum(F.col("event_id").isNull().cast("int"))
        .alias("null_event_id"),

    F.sum(F.col("actor_id").isNull().cast("int"))
        .alias("null_actor_id"),

    F.sum(F.col("repo_id").isNull().cast("int"))
        .alias("null_repo_id"),

    F.sum(F.col("action").isNull().cast("int"))
        .alias("null_action"),

    F.sum(F.col("issue_id").isNull().cast("int"))
        .alias("null_issue_id"),

    F.sum(F.col("issue_number").isNull().cast("int"))
        .alias("null_issue_number"),

    F.sum(F.col("state").isNull().cast("int"))
        .alias("null_state")
).show()

In [0]:
duplicate_count = (
    silver_issue_df
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate IssuesEvent IDs:",
    duplicate_count
)

In [0]:
if not spark.catalog.tableExists(SILVER_ISSUE_TABLE):
    (
        silver_issue_df.limit(0)
        .write
        .format("delta")
        .saveAsTable(SILVER_ISSUE_TABLE)
    )

silver_issue_delta = DeltaTable.forName(
    spark,
    SILVER_ISSUE_TABLE
)

(
    silver_issue_delta.alias("t")
    .merge(
        silver_issue_df.alias("s"),
        "t.event_id = s.event_id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("Silver Issue MERGE completed.")

In [0]:
print(
    "Silver Issue table rows:",
    spark.table(SILVER_ISSUE_TABLE).count()
)

print(
    "Duplicate IssuesEvent IDs:",
    spark.table(SILVER_ISSUE_TABLE)
        .groupBy("event_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)